# NuminaMath-1.5 embeddings with MathLeap-Qwen-8B

Embeds every `problem` of **AI-MO/NuminaMath-1.5** (train split, all 896,215 rows, dataset order, same pinned revision as
the BERT file) with **uw-math-ai/MathLeap-Qwen-8B** (Qwen3-Embedding-8B fine-tuned for math retrieval):
last-token pooling, L2-normalised, 4096 dimensions, the same vectors as `SentenceTransformer(model).encode(problem)`.

* **Document side, no prompt**: the model's `document` prompt is empty. Queries can be embedded later without a prompt
  (symmetric problem-to-problem search) or with the model's `query` prompt; the corpus file serves both.
* **Up to 512 tokens** (`MAX_SEQ_LENGTH`). The model's own default is 128 (its training length), which would cut 23% of
  NuminaMath problems, often dropping the actual question at the end. Set `MAX_SEQ_LENGTH = None` for the default.
* **Stored as float16** unit vectors: cosines are kept to about 1e-4, and the file is 7.3 GB instead of 14.7 GB.
* Row `i` of the `.npy` is row `i` of the dataset; the notebook checks this against `NuminaMath-1.5_rows.parquet`
  already in the repo (filter rows with that file).

**How it fits on Kaggle**: in float16 the 8B model is about 14 GiB, slightly more than a T4 can hold with room to work.
The token-embedding table (1.2 GB) stays in CPU memory, so each T4 holds a full copy and both GPUs work in parallel.
A reference check re-embeds a few rows with the official `SentenceTransformer` (CPU, bfloat16) and stops the run if they
do not match.

**Before running**
1. *Settings → Accelerator → GPU T4 x2*, *Internet on*.
2. *Add-ons → Secrets*: `HF_TOKEN` (write access) ticked for this notebook.
3. *Save Version → Save & Run All (Commit)*. Expected 6-10 hours. Every finished chunk is uploaded to
   `embeddings/MathLeap-Qwen-8B_parts/`; if the 12-hour limit is reached, run it again and it continues from there.

**Uploads to** `mostafabehroozi/numinamath-2025`, folder `embeddings/`:
* `embedding_NuminaMath-1.5_with_MathLeap-Qwen-8B.npy`, float16 `[896215, 4096]`, about 7.3 GB
* `NuminaMath-1.5_MathLeap-Qwen-8B_manifest.json`, the recipe, revisions, checks and timings

In [ ]:
# Settings
import time
from pathlib import Path

SESSION_T0 = time.time()

SOURCE_DATASET = "AI-MO/NuminaMath-1.5"
SOURCE_REVISION = "1b05109f9e5c1ad06c0663519502416c30b300f8"  # same rows as the BERT embedding file
SOURCE_SPLIT = "train"
TEXT_COLUMN = "problem"
MODEL_ID = "uw-math-ai/MathLeap-Qwen-8B"
MODEL_REVISION = None        # None = pin the current head (written to the manifest)

PROMPT = ""                  # document side: the model's "document" prompt is empty
MAX_SEQ_LENGTH = 512         # None = the model's default (128, its training length)
COMPUTE_DTYPE = "float16"    # T4 has no fast bfloat16
STORE_DTYPE = "float16"      # unit vectors: cosines kept to ~1e-4, half the size of float32

TARGET_REPO = "mostafabehroozi/numinamath-2025"   # dataset repo that receives the files
TARGET_DIR_IN_REPO = "embeddings"
EMBEDDING_FILE = "embedding_NuminaMath-1.5_with_MathLeap-Qwen-8B.npy"
MANIFEST_FILE = "NuminaMath-1.5_MathLeap-Qwen-8B_manifest.json"
ROWS_FILE = "NuminaMath-1.5_rows.parquet"          # already in the repo; proves the row order matches
PARTS_DIR_IN_REPO = f"{TARGET_DIR_IN_REPO}/MathLeap-Qwen-8B_parts"  # finished chunks, so a later session resumes
DELETE_PARTS_AFTER_MERGE = True

CHUNK_ROWS = 25_000          # one part file (about 205 MB) per chunk
TOKENS_PER_BATCH = 8192      # padded tokens per forward pass per GPU
MAX_BATCH = 256
RECIPE_CHECK_ROWS = 8        # rows re-embedded by the official SentenceTransformer on CPU as the reference
SPOT_CHECK_ROWS = 256        # rows re-embedded one by one after the run and compared with the stored rows
TIME_BUDGET_HOURS = 11.2     # no new chunk starts after this (Kaggle stops at 12 h); run again to continue
DELETE_MODEL_FILES_AFTER_LOAD = True   # frees ~30 GB of disk once both GPU copies and the reference check are done
WORK_DIR = Path("/tmp/mathleap_embed")  # outside /kaggle/working, so 7 GB is not saved as notebook output
UPLOAD_DIR = WORK_DIR / "upload"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Hugging Face token (Kaggle secret HF_TOKEN), an early write test, pinned revisions and a disk check
import datetime, json, os, shutil
from huggingface_hub import HfApi, hf_hub_download

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        raise RuntimeError("Add a Kaggle secret named HF_TOKEN (Add-ons -> Secrets) and tick it for this notebook.") from exc
os.environ["HF_TOKEN"] = HF_TOKEN

api = HfApi(token=HF_TOKEN)
HF_USER = api.whoami()["name"]
print("Hugging Face user:", HF_USER)
try:
    api.repo_info(TARGET_REPO, repo_type="dataset")
    print("Target repo exists:", TARGET_REPO)
except Exception:
    api.create_repo(TARGET_REPO, repo_type="dataset", private=True, exist_ok=True)
    print("Created private dataset repo:", TARGET_REPO)

STARTED_AT = datetime.datetime.now(datetime.timezone.utc).isoformat()
(WORK_DIR / "status.json").write_text(json.dumps(
    {"status": "running", "started_at": STARTED_AT, "source_dataset": SOURCE_DATASET, "model_id": MODEL_ID}, indent=2))
api.upload_file(path_or_fileobj=str(WORK_DIR / "status.json"), path_in_repo=f"{TARGET_DIR_IN_REPO}/{MANIFEST_FILE}",
                repo_id=TARGET_REPO, repo_type="dataset",
                commit_message="NuminaMath-1.5 MathLeap embeddings: run started (write test)")
print("Write access confirmed.")

assert api.dataset_info(SOURCE_DATASET, revision=SOURCE_REVISION).sha == SOURCE_REVISION
info = api.model_info(MODEL_ID, revision=MODEL_REVISION, files_metadata=True)
MODEL_REVISION = info.sha
MODEL_BYTES = sum((s.size or 0) for s in info.siblings)
print("Dataset revision:", SOURCE_REVISION)
print("Model revision:", MODEL_REVISION, f"| model files {MODEL_BYTES / 1e9:.1f} GB")

from huggingface_hub import constants as hf_constants
for p in ("/", "/tmp", "/kaggle/working", hf_constants.HF_HUB_CACHE):
    if os.path.exists(p):
        print(f"disk free at {p}: {shutil.disk_usage(p).free / 1e9:.0f} GB")
os.makedirs(hf_constants.HF_HUB_CACHE, exist_ok=True)
need = MODEL_BYTES + 8e9   # model files + the float16 output + working space
if shutil.disk_usage(hf_constants.HF_HUB_CACHE).free < need:
    raise RuntimeError(f"Not enough disk: need about {need / 1e9:.0f} GB free for the model files and the output.")

In [ ]:
# Load NuminaMath-1.5 at the pinned revision and check the row order against the rows file already in the repo
import hashlib
import numpy as np
import pandas as pd
from datasets import load_dataset

t0 = time.time()
ds = load_dataset(SOURCE_DATASET, split=SOURCE_SPLIT, revision=SOURCE_REVISION)
print(ds)
raw = ds[TEXT_COLUMN]
texts = [t if isinstance(t, str) else "" for t in raw]
EMPTY_ROWS = int(sum(1 for t in raw if not isinstance(t, str) or not t.strip()))
del raw, ds
N_ROWS = len(texts)
print(f"rows: {N_ROWS} | empty problems: {EMPTY_ROWS} | loaded in {time.time() - t0:.0f}s")

sha1 = np.array([hashlib.sha1(t.encode("utf-8")).hexdigest() for t in texts])
try:
    rows_path = hf_hub_download(TARGET_REPO, f"{TARGET_DIR_IN_REPO}/{ROWS_FILE}", repo_type="dataset", token=HF_TOKEN)
    ref = pd.read_parquet(rows_path, columns=["idx", "problem_sha1"])
except Exception as exc:
    ALIGNMENT = {"checked": False, "skipped_because": repr(exc)}
    print("Row-order check skipped:", repr(exc))
else:
    ok = len(ref) == N_ROWS and bool((ref["idx"].to_numpy() == np.arange(N_ROWS)).all()) \
        and bool((ref["problem_sha1"].to_numpy() == sha1).all())
    if not ok:
        raise RuntimeError(f"{ROWS_FILE} in the repo does not match the loaded rows: the row order would differ.")
    ALIGNMENT = {"checked": True, "rows_file": f"{TARGET_REPO}/{TARGET_DIR_IN_REPO}/{ROWS_FILE}", "matches": True}
    print("Row order matches", ROWS_FILE, "(every problem_sha1 equal).")

In [ ]:
# Model files, then one float16 copy per GPU; the token-embedding table stays in CPU memory
import gc, queue, threading
import torch
import torch.nn.functional as F
import transformers
from huggingface_hub import snapshot_download
from packaging.version import Version
from transformers import AutoConfig, AutoModel, AutoTokenizer

DTYPE_KW = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"
CDTYPE = getattr(torch, COMPUTE_DTYPE)

t0 = time.time()
MODEL_PATH = snapshot_download(MODEL_ID, revision=MODEL_REVISION, token=HF_TOKEN)
print(f"model files ready in {time.time() - t0:.0f}s: {MODEL_PATH}")
mp = Path(MODEL_PATH)
ST_MAX_SEQ = json.loads((mp / "sentence_bert_config.json").read_text())["max_seq_length"]
POOLING = json.loads((mp / "1_Pooling" / "config.json").read_text())
assert POOLING.get("pooling_mode_lasttoken"), f"this notebook implements last-token pooling only: {POOLING}"
MODULE_TYPES = [m["type"] for m in json.loads((mp / "modules.json").read_text())]
NORMALIZE = any(t.endswith("Normalize") for t in MODULE_TYPES)
ST_PROMPTS = json.loads((mp / "config_sentence_transformers.json").read_text()).get("prompts", {})
MAX_SEQ = MAX_SEQ_LENGTH or ST_MAX_SEQ
TOK = AutoTokenizer.from_pretrained(MODEL_PATH)
PAD_ID = TOK.pad_token_id if TOK.pad_token_id is not None else TOK.eos_token_id
LEFT_PAD = TOK.padding_side == "left"
DIM = AutoConfig.from_pretrained(MODEL_PATH).hidden_size
assert POOLING.get("word_embedding_dimension", DIM) == DIM
print(f"dim {DIM} | max tokens {MAX_SEQ} (model default {ST_MAX_SEQ}) | normalize {NORMALIZE} | "
      f"padding {TOK.padding_side} | prompts {ST_PROMPTS}")

DEVICES = [f"cuda:{i}" for i in range(torch.cuda.device_count())]
if not DEVICES:
    raise RuntimeError("No GPU: set Settings -> Accelerator -> GPU T4 x2.")
print("devices:", DEVICES, "| torch", torch.__version__, "| transformers", transformers.__version__)

WORKERS = []
for dev in DEVICES:
    t0 = time.time()
    m = AutoModel.from_pretrained(MODEL_PATH, **{DTYPE_KW: CDTYPE}, attn_implementation="sdpa")  # CPU, ~15 GB RAM
    m.eval()
    emb_mod = m.get_input_embeddings()
    for name, child in m.named_children():
        if child is not emb_mod:
            child.to(dev)
    for p in list(m.parameters(recurse=False)) + list(m.buffers(recurse=False)):
        p.data = p.data.to(dev)
    gc.collect()
    off = sorted({str(p.device) for p in m.parameters() if p is not emb_mod.weight} - {dev})
    assert not off, f"parameters left on {off}"
    free, total = torch.cuda.mem_get_info(dev)
    print(f"{dev}: loaded in {time.time() - t0:.0f}s | {free / 2**30:.2f} GiB free of {total / 2**30:.2f} GiB "
          f"| embedding table on {emb_mod.weight.device}")
    WORKERS.append({"device": dev, "model": m, "embed": emb_mod.weight})

In [ ]:
# Encoder: length-sorted token-budget batches, shared by the GPUs through a queue
TOKENS_DONE = 0


def tokenize(batch_texts):
    # SentenceTransformer prepends the prompt and strips each text before tokenising; the tokenizer appends <|endoftext|>
    return TOK([(PROMPT + t).strip() for t in batch_texts], truncation=True, max_length=MAX_SEQ)["input_ids"]


def make_batches(lengths):
    order = np.argsort(-np.asarray(lengths), kind="stable")
    batches, cur, cur_max = [], [], 0
    for i in order:
        new_max = max(cur_max, lengths[i])
        if cur and (new_max * (len(cur) + 1) > TOKENS_PER_BATCH or len(cur) >= MAX_BATCH):
            batches.append(cur)
            cur, new_max = [], lengths[i]
        cur.append(int(i))
        cur_max = new_max
    if cur:
        batches.append(cur)
    return batches


@torch.inference_mode()
def forward(worker, ids_list):
    dev, n, L = worker["device"], len(ids_list), max(len(x) for x in ids_list)
    ids = torch.full((n, L), PAD_ID, dtype=torch.long)
    mask = torch.zeros((n, L), dtype=torch.long)
    for r, x in enumerate(ids_list):
        sl = slice(L - len(x), L) if LEFT_PAD else slice(0, len(x))
        ids[r, sl] = torch.tensor(x, dtype=torch.long)
        mask[r, sl] = 1
    x = F.embedding(ids, worker["embed"]).to(dev, non_blocking=True)
    hs = worker["model"](inputs_embeds=x, attention_mask=mask.to(dev), use_cache=False).last_hidden_state
    last = torch.full((n,), L - 1, dtype=torch.long) if LEFT_PAD else mask.sum(1) - 1
    h = hs[torch.arange(n, device=dev), last.to(dev)].float()
    if NORMALIZE:
        h = F.normalize(h, p=2, dim=1)
    return h.cpu().numpy()


def forward_safe(worker, ids_list):
    try:
        return forward(worker, ids_list)
    except torch.cuda.OutOfMemoryError:
        if len(ids_list) == 1:
            raise
        torch.cuda.empty_cache()
        h = len(ids_list) // 2
        return np.concatenate([forward_safe(worker, ids_list[:h]), forward_safe(worker, ids_list[h:])])


def encode(batch_texts, workers=None):
    """float32 [n, DIM], input order; equals SentenceTransformer(model).encode(PROMPT + text) up to float noise."""
    global TOKENS_DONE
    workers = workers or WORKERS
    ids = tokenize(list(batch_texts))
    out = np.zeros((len(ids), DIM), np.float32)
    jobs = queue.Queue()
    for b in make_batches([len(x) for x in ids]):
        jobs.put(b)
    errors = []

    def run(w):
        try:
            while True:
                try:
                    b = jobs.get_nowait()
                except queue.Empty:
                    return
                out[b] = forward_safe(w, [ids[i] for i in b])
        except BaseException as exc:  # surfaced in the main thread
            errors.append(exc)

    threads = [threading.Thread(target=run, args=(w,)) for w in workers]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    if errors:
        raise errors[0]
    TOKENS_DONE += sum(len(x) for x in ids)
    return out


def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)


t0 = time.time()
warm = encode(texts[:64])
print(f"warm-up: 64 rows in {time.time() - t0:.1f}s, finite {bool(np.isfinite(warm).all())}")

In [ ]:
# Reference check: the official SentenceTransformer (CPU, bfloat16) on a few random rows must give the same vectors
class RecipeMismatch(RuntimeError):
    pass


from sentence_transformers import SentenceTransformer
import sentence_transformers

rng = np.random.default_rng(1)
ref_idx = np.sort(rng.choice(N_ROWS, size=RECIPE_CHECK_ROWS, replace=False))
ref_texts = [texts[i] for i in ref_idx]
per_gpu = [encode(ref_texts, [w]) for w in WORKERS]
mine = encode(ref_texts)
gpu_agreement = min(float((unit(per_gpu[0]) * unit(p)).sum(1).min()) for p in per_gpu + [mine])

t0 = time.time()
torch.set_num_threads(os.cpu_count() or 4)
st = SentenceTransformer(MODEL_PATH, device="cpu", model_kwargs={DTYPE_KW: torch.bfloat16})
st.max_seq_length = MAX_SEQ
ref = st.encode([PROMPT + t for t in ref_texts], batch_size=len(ref_texts), convert_to_numpy=True,
                show_progress_bar=False).astype(np.float32)
del st
gc.collect()
cos = (unit(mine) * unit(ref)).sum(1)
RECIPE_RESULT = {"rows": ref_idx.tolist(), "reference": "SentenceTransformer(model, device='cpu', bfloat16).encode",
                 "cosine_min": float(cos.min()), "cosine_mean": float(cos.mean()),
                 "gpu_copies_agree_cosine_min": gpu_agreement, "reference_seconds": round(time.time() - t0, 1),
                 "token_lengths": [len(x) for x in tokenize(ref_texts)]}
print(json.dumps(RECIPE_RESULT, indent=2))
if not (np.isfinite(mine).all() and cos.min() >= 0.98 and gpu_agreement >= 0.999):
    raise RecipeMismatch("Reference check FAILED: the GPU float16 vectors differ from the official encoder. Stopping.")
print("Reference check passed (bfloat16 vs float16 noise only).")

repo_cache = mp.parents[1]  # .../models--<org>--<name>/snapshots/<sha>
if DELETE_MODEL_FILES_AFTER_LOAD and repo_cache.name.startswith("models--"):
    shutil.rmtree(repo_cache, ignore_errors=True)  # the GPUs keep their copies
    print(f"model files deleted | disk free {shutil.disk_usage(WORK_DIR).free / 1e9:.0f} GB")

In [ ]:
# Embed chunk by chunk; each finished chunk is uploaded as a part, so a later session resumes from Hugging Face
from concurrent.futures import ThreadPoolExecutor
from numpy.lib.format import open_memmap

SDT = np.dtype(STORE_DTYPE)
n_chunks = (N_ROWS + CHUNK_ROWS - 1) // CHUNK_ROWS
PARTS_META = {"source_dataset": SOURCE_DATASET, "source_revision": SOURCE_REVISION, "rows": N_ROWS,
              "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "max_seq_length": MAX_SEQ, "prompt": PROMPT,
              "compute_dtype": COMPUTE_DTYPE, "store_dtype": STORE_DTYPE, "dim": DIM, "chunk_rows": CHUNK_ROWS}


def chunk_span(c):
    return c * CHUNK_ROWS, min(N_ROWS, (c + 1) * CHUNK_ROWS)


def part_path(c):
    return f"{PARTS_DIR_IN_REPO}/part_{c:04d}.npy"


try:
    remote_parts = {f.path for f in api.list_repo_tree(TARGET_REPO, path_in_repo=PARTS_DIR_IN_REPO,
                                                       repo_type="dataset")}
except Exception:
    remote_parts = set()
meta_file = f"{PARTS_DIR_IN_REPO}/parts_meta.json"
if meta_file in remote_parts:
    old_meta = json.loads(Path(hf_hub_download(TARGET_REPO, meta_file, repo_type="dataset", token=HF_TOKEN)).read_text())
    if old_meta != PARTS_META:
        raise RuntimeError(f"Parts in {PARTS_DIR_IN_REPO} were made with other settings:\n{old_meta}\nvs\n{PARTS_META}\n"
                           "Restore those settings, or delete that folder to start over.")
else:
    (WORK_DIR / "parts_meta.json").write_text(json.dumps(PARTS_META, indent=2))
    api.upload_file(path_or_fileobj=str(WORK_DIR / "parts_meta.json"), path_in_repo=meta_file, repo_id=TARGET_REPO,
                    repo_type="dataset", commit_message="MathLeap embedding parts: settings")

EMB_PATH = UPLOAD_DIR / EMBEDDING_FILE
emb = open_memmap(EMB_PATH, mode="w+", dtype=SDT, shape=(N_ROWS, DIM))
done = set()
for c in range(n_chunks):
    if part_path(c) not in remote_parts:
        continue
    p = hf_hub_download(TARGET_REPO, part_path(c), repo_type="dataset", token=HF_TOKEN, local_dir=str(WORK_DIR / "dl"))
    a = np.load(p)
    s, e = chunk_span(c)
    if a.shape == (e - s, DIM) and a.dtype == SDT and np.isfinite(a).all():
        emb[s:e] = a
        done.add(c)
    os.remove(p)
print(f"{len(done)}/{n_chunks} chunks already on Hugging Face")

UPLOADS = ThreadPoolExecutor(1)
pending = []


def upload_part(c, path):
    for attempt in range(5):
        try:
            api.upload_file(path_or_fileobj=str(path), path_in_repo=part_path(c), repo_id=TARGET_REPO,
                            repo_type="dataset", commit_message=f"MathLeap embeddings part {c + 1}/{n_chunks}")
            path.unlink()
            return
        except Exception:
            if attempt == 4:
                raise
            time.sleep(30 * (attempt + 1))


t0, rows_now, tok0 = time.time(), 0, TOKENS_DONE
for c in range(n_chunks):
    if c in done:
        continue
    if (time.time() - SESSION_T0) / 3600 > TIME_BUDGET_HOURS:
        print(f"time budget of {TIME_BUDGET_HOURS} h reached; stopping before chunk {c + 1}")
        break
    s, e = chunk_span(c)
    a = encode(texts[s:e]).astype(SDT)
    if not np.isfinite(a).all():
        raise RuntimeError(f"NaN/inf in chunk {c} (float16 overflow?)")
    emb[s:e] = a
    pp = WORK_DIR / f"part_{c:04d}.npy"
    np.save(pp, a)
    pending.append(UPLOADS.submit(upload_part, c, pp))
    for f in pending:
        if f.done() and f.exception():
            raise f.exception()
    done.add(c)
    rows_now += e - s
    el = time.time() - t0
    left = sum(chunk_span(k)[1] - chunk_span(k)[0] for k in range(n_chunks) if k not in done)
    print(f"chunk {c + 1}/{n_chunks} rows {s}-{e} | {rows_now / el:.1f} rows/s, {(TOKENS_DONE - tok0) / el:.0f} tok/s "
          f"| ETA {left / (rows_now / el) / 3600:.2f} h", flush=True)
for f in pending:
    f.result()
emb.flush()
del emb
EMBED_SECONDS = time.time() - t0
if len(done) < n_chunks:
    raise RuntimeError(f"Stopped at the time budget: {len(done)}/{n_chunks} chunks are saved in {PARTS_DIR_IN_REPO}. "
                       "Run the notebook again (Save & Run All) to continue from there.")
print(f"all {n_chunks} chunks done; this session embedded {rows_now} rows in {EMBED_SECONDS / 3600:.2f} h")

In [ ]:
# Checks: shape, no NaN/zero rows, unit norms, and a one-by-one re-embedding of random rows matches the stored rows
emb = np.load(EMB_PATH, mmap_mode="r")
assert emb.shape == (N_ROWS, DIM) and emb.dtype == SDT, (emb.shape, emb.dtype)
bad, zero_rows, norms = 0, 0, []
for s in range(0, N_ROWS, 50_000):
    block = np.asarray(emb[s:s + 50_000], np.float32)
    bad += int((~np.isfinite(block)).any(1).sum())
    nb = np.linalg.norm(block, axis=1)
    zero_rows += int((nb < 1e-6).sum())
    norms.append(nb)
norms = np.concatenate(norms)
assert bad == 0 and zero_rows == 0, (bad, zero_rows)
if NORMALIZE:
    assert np.abs(norms - 1).max() < 1e-2, (norms.min(), norms.max())

rng = np.random.default_rng(0)
sample = np.sort(rng.choice(N_ROWS, size=min(SPOT_CHECK_ROWS, N_ROWS), replace=False))
fresh = np.stack([encode([texts[i]], WORKERS[:1])[0] for i in sample])
spot = (unit(fresh) * unit(np.asarray(emb[sample], np.float32))).sum(1)
CHECKS = {"nan_rows": bad, "zero_norm_rows": zero_rows, "norm_min": float(norms.min()),
          "norm_max": float(norms.max()), "spot_check_rows": int(len(sample)),
          "spot_check_cosine_min": float(spot.min()), "spot_check_cosine_mean": float(spot.mean())}
print(json.dumps(CHECKS, indent=2))
assert spot.min() > 0.999, "stored rows do not match a fresh embedding of the same problem"
del emb

In [ ]:
# Manifest, upload of the merged file, then the parts are removed
sha = hashlib.sha256()
with open(EMB_PATH, "rb") as fh:
    for blk in iter(lambda: fh.read(1 << 24), b""):
        sha.update(blk)

manifest = {
    "status": "complete", "started_at": STARTED_AT,
    "finished_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "source_dataset": SOURCE_DATASET, "source_revision": SOURCE_REVISION, "split": SOURCE_SPLIT,
    "text_column": TEXT_COLUMN, "rows": N_ROWS, "empty_text_rows": EMPTY_ROWS,
    "row_order": "row i of the .npy is row i of the split at source_revision (same as NuminaMath-1.5_rows.parquet)",
    "row_order_check": ALIGNMENT,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "dim": DIM,
    "max_seq_length": MAX_SEQ, "model_default_max_seq_length": ST_MAX_SEQ,
    "pooling": "last token (the model's sentence-transformers Pooling module)", "normalized": NORMALIZE,
    "prompt": PROMPT, "model_prompts": ST_PROMPTS,
    "compute_dtype": COMPUTE_DTYPE, "dtype": STORE_DTYPE,
    "recipe": f"m = SentenceTransformer('{MODEL_ID}'); m.max_seq_length = {MAX_SEQ}; "
              f"m.encode(problems)  (document side, no prompt; L2-normalised) -> stored as {STORE_DTYPE}",
    "queries": "embed with the same model and max_seq_length; no prompt for problem-to-problem similarity, or the "
               "model's 'query' prompt for query-to-document retrieval; similarity = dot product (unit vectors)",
    "devices": DEVICES, "tokens_per_batch": TOKENS_PER_BATCH, "last_session_embed_hours": round(EMBED_SECONDS / 3600, 3),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "sentence_transformers": sentence_transformers.__version__,
    "embedding_sha256": sha.hexdigest(), "embedding_bytes": EMB_PATH.stat().st_size,
    "reference_check": RECIPE_RESULT, "checks": CHECKS,
    "files": {"embeddings": EMBEDDING_FILE, "rows": ROWS_FILE},
}
(UPLOAD_DIR / MANIFEST_FILE).write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

t0 = time.time()
api.upload_folder(folder_path=str(UPLOAD_DIR), path_in_repo=TARGET_DIR_IN_REPO, repo_id=TARGET_REPO,
                  repo_type="dataset",
                  commit_message=f"NuminaMath-1.5 problem embeddings ({N_ROWS} x {DIM}, {MODEL_ID}, "
                                 f"max {MAX_SEQ} tokens, rev {SOURCE_REVISION[:8]})")
print(f"uploaded in {time.time() - t0:.0f}s")
remote = {f.path: getattr(f, "size", None)
          for f in api.list_repo_tree(TARGET_REPO, path_in_repo=TARGET_DIR_IN_REPO, repo_type="dataset")}
print(json.dumps(remote, indent=2))
assert remote.get(f"{TARGET_DIR_IN_REPO}/{EMBEDDING_FILE}") == EMB_PATH.stat().st_size, "uploaded size differs"
if DELETE_PARTS_AFTER_MERGE:
    api.delete_folder(PARTS_DIR_IN_REPO, repo_id=TARGET_REPO, repo_type="dataset",
                      commit_message="MathLeap embeddings: remove parts (merged file uploaded)")
    print("parts removed:", PARTS_DIR_IN_REPO)
print(f"\nDONE: https://huggingface.co/datasets/{TARGET_REPO}/tree/main/{TARGET_DIR_IN_REPO}")